In [ ]:
from pyGmsh import pyGmsh
import numpy as np
import openseespy.opensees as ops
import gmsh
import matplotlib.pyplot as plt

# Nonlinear Buckling Analysis — IPE200 with Initial Imperfection

**Goal:** Use the first buckling mode shape as a geometric imperfection
and run a geometrically nonlinear analysis to trace the full
load–displacement path through and beyond the critical load.

**Methodology — imperfection-based nonlinear analysis:**

1. **Linear eigenbuckling** → extract the critical mode shape (LTB mode).
2. **Scale the mode shape** → apply as a small geometric imperfection
   (perturb node coordinates by amplitude ≈ L/1000).
3. **Nonlinear static analysis** → displacement-controlled, traces the
   equilibrium path including pre-buckling, limit point, and
   post-buckling regimes.
4. **Load–displacement curve** → reveals the actual collapse load,
   which for imperfect structures is always lower than the theoretical
   bifurcation load from linear buckling.

**Why this matters:**

Real structures are never geometrically perfect. Eurocode 3 and AISC
360 require engineers to account for initial imperfections when checking
stability. The imperfection-based nonlinear approach gives a realistic
collapse load without relying on effective length factors or moment
gradient coefficients.

**Physics captured by the shell model:**
- Geometric nonlinearity (large displacements, follower effects)
- Coupling between bending, lateral displacement, and twist (LTB)
- Local flange/web distortion near the buckled zone
- Imperfection sensitivity (how the collapse load depends on imperfection amplitude)

In [ ]:
# ============================================================
# IPE200 cross-section & material
# ============================================================
h   = 200.0     # [mm]
b   = 100.0     # [mm]
tw  = 5.6       # [mm]
tf  = 8.5       # [mm]
L   = 3000.0    # [mm]
d   = h - tf    # 191.5 mm  (distance between flange mid-surfaces)

E   = 210000.0  # [MPa]
nu  = 0.3
rho = 7.85e-9   # [tonne/mm³]
G   = E / (2*(1 + nu))

# Section properties (for analytical reference)
A_cs  = 2*b*tf + (h - 2*tf)*tw
Iy_cs = (b*h**3 - (b - tw)*(h - 2*tf)**3) / 12
Iz_cs = (2*tf*b**3 + (h - 2*tf)*tw**3) / 12
J_cs  = (2*b*tf**3 + (h - 2*tf)*tw**3) / 3

# Imperfection amplitude
# Eurocode 3 recommends L/1000 for global bow imperfection
imp_amplitude = L / 1000.0   # [mm] = 3.0 mm

# Mesh divisions
n_half_flange = 4
n_web_height  = 12
n_length      = 60

print(f"IPE200 cantilever: L = {L:.0f} mm")
print(f"Imperfection amplitude: {imp_amplitude:.1f} mm  (L/{int(L/imp_amplitude)})")

## Part 1 — Geometry, Mesh & Buckling Mode Extraction

We reuse the same 5-surface mid-surface I-beam geometry. The workflow:

1. Build the **perfect** geometry and mesh with pyGmsh
2. Create a temporary **perfect** OpenSees model
3. Run free-vibration `ops.eigen()` to extract mode shapes
4. Identify the **LTB mode** (dominant lateral displacement)
5. `ops.wipe()` — the perfect model is discarded
6. Scale the LTB mode shape → **geometric imperfection**

In [ ]:
g = pyGmsh(model_name="IPE200_NL_Buckling", verbose=True)
g.initialize()

# --- 12 corner points ---
p1  = g.model.add_point(0, -b/2, 0,  lc=30)
p2  = g.model.add_point(0,  0,   0,  lc=30)
p3  = g.model.add_point(0,  b/2, 0,  lc=30)
p4  = g.model.add_point(0,  0,   d,  lc=30)
p5  = g.model.add_point(0, -b/2, d,  lc=30)
p6  = g.model.add_point(0,  b/2, d,  lc=30)

p7  = g.model.add_point(L, -b/2, 0,  lc=30)
p8  = g.model.add_point(L,  0,   0,  lc=30)
p9  = g.model.add_point(L,  b/2, 0,  lc=30)
p10 = g.model.add_point(L,  0,   d,  lc=30)
p11 = g.model.add_point(L, -b/2, d,  lc=30)
p12 = g.model.add_point(L,  b/2, d,  lc=30)

# --- Lines ---
c1 = g.model.add_line(p1, p2);  c2 = g.model.add_line(p2, p3)
c3 = g.model.add_line(p2, p4);  c4 = g.model.add_line(p5, p4)
c5 = g.model.add_line(p4, p6)

c6  = g.model.add_line(p7,  p8);   c7  = g.model.add_line(p8,  p9)
c8  = g.model.add_line(p8,  p10);  c9  = g.model.add_line(p11, p10)
c10 = g.model.add_line(p10, p12)

e1 = g.model.add_line(p1, p7);   e2 = g.model.add_line(p2, p8)
e3 = g.model.add_line(p3, p9);   e4 = g.model.add_line(p4, p10)
e5 = g.model.add_line(p5, p11);  e6 = g.model.add_line(p6, p12)

# --- 5 Surfaces ---
s_bf_l = g.model.add_plane_surface(
    g.model.add_curve_loop([c1, e2, -c6, -e1]))
s_bf_r = g.model.add_plane_surface(
    g.model.add_curve_loop([c2, e3, -c7, -e2]))
s_web  = g.model.add_plane_surface(
    g.model.add_curve_loop([c3, e4, -c8, -e2]))
s_tf_l = g.model.add_plane_surface(
    g.model.add_curve_loop([-c4, e5, c9, -e4]))
s_tf_r = g.model.add_plane_surface(
    g.model.add_curve_loop([c5, e6, -c10, -e4]))

# --- Physical groups ---
pg_flanges = g.physical.add(2, [s_bf_l, s_bf_r, s_tf_l, s_tf_r], name="Flanges")
pg_web     = g.physical.add(2, [s_web], name="Web")
pg_pin     = g.physical.add(1, [c1, c2, c3, c4, c5], name="PinEnd")
pg_roller  = g.physical.add(1, [c6, c7, c8, c9, c10], name="RollerEnd")

# --- Transfinite quad mesh ---
for c in [c1, c2, c4, c5, c6, c7, c9, c10]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_half_flange + 1)
for c in [c3, c8]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_web_height + 1)
for c in [e1, e2, e3, e4, e5, e6]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_length + 1)
for s in [s_bf_l, s_bf_r, s_web, s_tf_l, s_tf_r]:
    gmsh.model.mesh.setTransfiniteSurface(s)
    gmsh.model.mesh.setRecombine(2, s)

g.mesh.set_order(1)
g.mesh.generate(2)

print(f"Mesh generated: {sum(len(t) for t in g.mesh.get_elements(dim=2)['tags'])} quad elements")

In [ ]:
# --- Extract mesh data ---
fem = g.mesh.get_fem_data(dim=2)
node_tags    = fem['node_tags']
node_coords  = fem['node_coords']     # ORIGINAL (perfect) coordinates
tag_to_idx   = fem['tag_to_idx']
connectivity = fem['connectivity']
elem_tags    = fem['elem_tags']
used_tags    = fem['used_tags']

# Region classification
flange_elem_set = set()
for surf in [s_bf_l, s_bf_r, s_tf_l, s_tf_r]:
    for etags in g.mesh.get_elements(dim=2, tag=surf)['tags']:
        flange_elem_set.update(etags.astype(int).tolist())

web_elem_set = set()
for etags in g.mesh.get_elements(dim=2, tag=s_web)['tags']:
    web_elem_set.update(etags.astype(int).tolist())

# Boundary nodes
pin_nodes    = g.physical.get_nodes(1, pg_pin)['tags']
roller_nodes = g.physical.get_nodes(1, pg_roller)['tags']

# Load node (midspan, top of web)
target = np.array([L/2, 0.0, d])
dists  = np.linalg.norm(node_coords - target, axis=1)
load_idx  = np.argmin(dists)
load_gtag = int(node_tags[load_idx])

# Lateral tracking node (midspan, top flange tip — where LTB is most visible)
target_lat = np.array([L/2, -b/2, d])
dists_lat  = np.linalg.norm(node_coords - target_lat, axis=1)
lat_idx    = np.argmin(dists_lat)
lat_gtag   = int(node_tags[lat_idx])

print(f"Nodes: {len(used_tags)},  Elements: {connectivity.shape[0]}")
print(f"Load node: tag {load_gtag} at {node_coords[load_idx]}")
print(f"Lateral tracking node: tag {lat_gtag} at {node_coords[lat_idx]}")

In [ ]:
# --- Build temporary PERFECT model for eigenbuckling ---
#
# We only need the eigenvectors (mode shapes), not the eigenvalues.
# Free vibration on the unloaded model gives us the LTB mode shape
# which we will use as the imperfection pattern.

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

gmsh_to_ops_tmp = {}
new_id = 0
for gtag, coords in zip(node_tags.astype(int), node_coords):
    if int(gtag) not in used_tags:
        continue
    new_id += 1
    gmsh_to_ops_tmp[int(gtag)] = new_id
    ops.node(new_id, float(coords[0]), float(coords[1]), float(coords[2]))

ops.section('ElasticMembranePlateSection', 1, E, nu, tf, rho)
ops.section('ElasticMembranePlateSection', 2, E, nu, tw, rho)

for i, (etag, row) in enumerate(zip(elem_tags, connectivity)):
    eid = i + 1
    nodes = [gmsh_to_ops_tmp[int(n)] for n in row]
    sec = 1 if etag in flange_elem_set else 2
    ops.element('ShellMITC4', eid, *nodes, sec)

for gtag in pin_nodes.astype(int):
    if int(gtag) in gmsh_to_ops_tmp:
        ops.fix(gmsh_to_ops_tmp[int(gtag)], 1, 1, 1, 0, 0, 0)
for gtag in roller_nodes.astype(int):
    if int(gtag) in gmsh_to_ops_tmp:
        ops.fix(gmsh_to_ops_tmp[int(gtag)], 0, 1, 1, 0, 0, 0)

# Eigenvalue analysis on perfect model
eigenvalues = ops.eigen(10)
print("Free vibration on PERFECT geometry:")
for i, ev in enumerate(eigenvalues):
    print(f"  Mode {i+1}: f = {np.sqrt(ev)/(2*np.pi):.3f} Hz")

In [ ]:
# --- Identify the LTB mode ---
#
# LTB = lateral (u_y) + torsional (θ_x) mode.
# We find the mode with the largest RMS lateral displacement.

nNode = len(node_tags)
mode_shapes = []
rms_uy = []

for m in range(1, 11):
    phi = np.zeros((nNode, 6))
    for gtag, ops_id in gmsh_to_ops_tmp.items():
        idx = tag_to_idx[gtag]
        for dof in range(6):
            phi[idx, dof] = ops.nodeEigenvector(ops_id, m, dof + 1)
    mode_shapes.append(phi)
    rms_uy.append(np.sqrt(np.mean(phi[:, 1]**2)))

# The LTB mode has the largest lateral (uy) component
ltb_mode_idx = np.argmax(rms_uy)
ltb_shape    = mode_shapes[ltb_mode_idx]  # (nNode, 6)

print(f"LTB mode identified: Mode {ltb_mode_idx + 1}")
print(f"  RMS(uy) = {rms_uy[ltb_mode_idx]:.6f}")
print(f"  Frequency = {np.sqrt(eigenvalues[ltb_mode_idx])/(2*np.pi):.3f} Hz")

# Clean up temporary model
ops.wipe()

## Part 2 — Geometric Imperfection

We scale the LTB mode shape so that the maximum lateral (y) displacement
equals the chosen imperfection amplitude:

$$\mathbf{X}_{\text{imperfect}} = \mathbf{X}_{\text{perfect}} + \alpha \cdot \boldsymbol{\phi}_{\text{LTB}}$$

where α is chosen so that max|φ_y| × α = L/1000.

The imperfection introduces a tiny asymmetry that **seeds** the
lateral-torsional instability. Without it, the perfect model would
follow the primary (in-plane bending) equilibrium path indefinitely
and never bifurcate — the nonlinear solver would miss the buckling
entirely.

In [ ]:
# --- Scale and apply imperfection ---

# Translational components of the LTB mode shape
phi_trans = ltb_shape[:, :3]   # (nNode, 3) → ux, uy, uz

# Scale so max lateral displacement = imp_amplitude
max_uy = np.max(np.abs(phi_trans[:, 1]))
if max_uy > 1e-15:
    scale_factor = imp_amplitude / max_uy
else:
    raise ValueError("LTB mode has zero lateral displacement — check mode identification")

# Perturbed coordinates
node_coords_imp = node_coords.copy()
node_coords_imp += scale_factor * phi_trans

# Report the perturbation
delta = node_coords_imp - node_coords
print(f"Imperfection applied (Mode {ltb_mode_idx + 1}):")
print(f"  Scale factor: {scale_factor:.4f}")
print(f"  Max Δy (lateral): {np.max(np.abs(delta[:, 1])):.3f} mm")
print(f"  Max Δx (axial):   {np.max(np.abs(delta[:, 0])):.3f} mm")
print(f"  Max Δz (vertical):{np.max(np.abs(delta[:, 2])):.3f} mm")

# --- Quick visualization: cross-section at midspan ---
fig, ax = plt.subplots(figsize=(8, 6))

tol_x = 30  # mm tolerance around midspan
mid_mask = np.abs(node_coords[:, 0] - L/2) < tol_x

# Perfect cross-section
ax.plot(node_coords[mid_mask, 1], node_coords[mid_mask, 2],
        'b.', markersize=3, alpha=0.3, label='Perfect')

# Imperfect cross-section (magnified for visibility)
vis_scale = 20.0  # visual magnification for the plot
coords_vis = node_coords + vis_scale * scale_factor * phi_trans
ax.plot(coords_vis[mid_mask, 1], coords_vis[mid_mask, 2],
        'r.', markersize=3, alpha=0.6, label=f'Imperfect (×{vis_scale:.0f})')

ax.set_xlabel('Y [mm]')
ax.set_ylabel('Z [mm]')
ax.set_title(f'Midspan cross-section — imperfection shape (magnified ×{vis_scale:.0f})')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3 — Nonlinear Analysis

**Strategy: displacement-controlled loading**

We impose incremental vertical displacement (u_z) at the midspan load
point and let OpenSees find the corresponding load factor. This naturally
traces the load–displacement curve, including the **limit point** where
the load reaches its maximum (the nonlinear buckling load) and the
**post-buckling** descending branch.

Why displacement control instead of load control?
- Load control cannot pass through a **limit point** (the analysis would
  diverge when dP/dδ → 0).
- Displacement control on u_z works as long as δ_z increases monotonically,
  which it does for LTB (the beam keeps deflecting vertically even after
  it buckles laterally).
- For snap-back behaviour (unlikely for LTB), arc-length would be needed.

**Key outputs:**
- P vs δ_z (load–displacement curve at midspan)
- u_y vs δ_z (lateral displacement growth — reveals onset of LTB)

In [ ]:
# --- Build OpenSees model with IMPERFECT geometry ---

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

gmsh_to_ops = {}
new_id = 0
for gtag, coords_imp in zip(node_tags.astype(int), node_coords_imp):
    if int(gtag) not in used_tags:
        continue
    new_id += 1
    gmsh_to_ops[int(gtag)] = new_id
    # NOTE: using IMPERFECT coordinates
    ops.node(new_id, float(coords_imp[0]), float(coords_imp[1]), float(coords_imp[2]))

# Sections (elastic — geometric nonlinearity only)
ops.section('ElasticMembranePlateSection', 1, E, nu, tf, rho)
ops.section('ElasticMembranePlateSection', 2, E, nu, tw, rho)

# Elements
for i, (etag, row) in enumerate(zip(elem_tags, connectivity)):
    eid = i + 1
    nodes = [gmsh_to_ops[int(n)] for n in row]
    sec = 1 if etag in flange_elem_set else 2
    ops.element('ShellMITC4', eid, *nodes, sec)

# Simply-supported BCs
for gtag in pin_nodes.astype(int):
    if int(gtag) in gmsh_to_ops:
        ops.fix(gmsh_to_ops[int(gtag)], 1, 1, 1, 0, 0, 0)
for gtag in roller_nodes.astype(int):
    if int(gtag) in gmsh_to_ops:
        ops.fix(gmsh_to_ops[int(gtag)], 0, 1, 1, 0, 0, 0)

# Load pattern: unit vertical load at midspan
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
load_ops = gmsh_to_ops[load_gtag]
lat_ops  = gmsh_to_ops[lat_gtag]
ops.load(load_ops, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0)

print(f"Nonlinear model built with imperfect geometry")
print(f"  {new_id} nodes, {connectivity.shape[0]} ShellMITC4 elements")
print(f"  Load node (ops): {load_ops}")
print(f"  Lateral tracking node (ops): {lat_ops}")

In [ ]:
# --- Displacement-controlled nonlinear analysis ---
#
# We push the midspan down in small increments and record:
#   - Load factor λ (= applied force in N, since reference load is 1 N)
#   - Vertical displacement u_z at midspan
#   - Lateral displacement u_y at top flange tip (LTB indicator)

dz_incr   = -0.2     # [mm] vertical displacement increment per step (downward)
max_steps  = 250      # enough to go well past the limit point
target_uz  = -50.0    # [mm] stop if midspan deflection exceeds this

# Analysis setup
ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-6, 50)
ops.algorithm('Newton')

# Storage
load_history = [0.0]          # load factor [N]
uz_history   = [0.0]          # vertical disp at midspan [mm]
uy_history   = [0.0]          # lateral disp at flange tip [mm]
step_ok      = True

print(f"Running nonlinear analysis: Δu_z = {dz_incr} mm/step, max {max_steps} steps")
print(f"{'Step':>5s}  {'P [kN]':>9s}  {'u_z [mm]':>9s}  {'u_y [mm]':>9s}")
print("-" * 38)

for step in range(1, max_steps + 1):
    ops.integrator('DisplacementControl', load_ops, 3, dz_incr)
    ops.analysis('Static')
    ok = ops.analyze(1)
    
    if ok != 0:
        # Try Newton with line search if standard Newton fails
        ops.algorithm('NewtonLineSearch', '-type', 'Bisection')
        ok = ops.analyze(1)
        ops.algorithm('Newton')
    
    if ok != 0:
        # Try ModifiedNewton as last resort
        ops.algorithm('ModifiedNewton')
        ok = ops.analyze(1)
        ops.algorithm('Newton')
    
    if ok != 0:
        print(f"  Step {step}: FAILED TO CONVERGE — stopping")
        break
    
    lam = ops.getTime()              # load factor = total applied load [N]
    uz  = ops.nodeDisp(load_ops, 3)  # vertical disp [mm]
    uy  = ops.nodeDisp(lat_ops, 2)   # lateral disp [mm]
    
    load_history.append(lam)
    uz_history.append(uz)
    uy_history.append(uy)
    
    if step % 25 == 0 or step <= 5:
        print(f"{step:5d}  {lam/1000:9.2f}  {uz:9.3f}  {uy:9.4f}")
    
    if abs(uz) > abs(target_uz):
        print(f"  Reached target deflection — stopping at step {step}")
        break

load_history = np.array(load_history)
uz_history   = np.array(uz_history)
uy_history   = np.array(uy_history)

# Identify the limit point (maximum load)
P_max_idx = np.argmax(load_history)
P_max = load_history[P_max_idx]
uz_at_Pmax = uz_history[P_max_idx]
uy_at_Pmax = uy_history[P_max_idx]

print(f"\n{'='*50}")
print(f"LIMIT POINT (nonlinear buckling load):")
print(f"  P_max  = {P_max/1000:.2f} kN")
print(f"  u_z    = {uz_at_Pmax:.3f} mm")
print(f"  u_y    = {uy_at_Pmax:.4f} mm")
print(f"  at step {P_max_idx}")

In [ ]:
# --- Extract final deformed shape (at last converged step) ---

disp_final = np.zeros((nNode, 6))
for gtag, ops_id in gmsh_to_ops.items():
    idx = tag_to_idx[gtag]
    for dof in range(6):
        disp_final[idx, dof] = ops.nodeDisp(ops_id, dof + 1)

ops.wipe()

print(f"Final state: P = {load_history[-1]/1000:.2f} kN, "
      f"u_z = {uz_history[-1]:.2f} mm, u_y = {uy_history[-1]:.4f} mm")

## Part 4 — Results

The load–displacement curve reveals three distinct regimes:

1. **Pre-buckling** (P < ~0.8 P_max): nearly linear, in-plane bending
   dominates. Lateral displacement u_y is negligible.
2. **Transition** (~0.8–1.0 P_max): lateral displacement grows rapidly
   as the geometric nonlinearity amplifies the imperfection.
3. **Post-buckling** (P decreasing): the beam has buckled laterally;
   increasing vertical deflection is accompanied by large lateral sway.

The **limit point** P_max is the nonlinear buckling load — always lower
than the linear bifurcation load P_cr from the VCT analysis because the
imperfection rounds off the sharp bifurcation into a smooth limit point.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) P vs u_z (primary load-displacement curve)
ax = axes[0]
ax.plot(-uz_history, load_history/1000, 'b-', lw=1.5)
ax.plot(-uz_at_Pmax, P_max/1000, 'ro', markersize=8, zorder=5,
        label=f'P_max = {P_max/1000:.2f} kN')
ax.set_xlabel('−u_z [mm] (downward deflection)')
ax.set_ylabel('P [kN]')
ax.set_title('Load–displacement curve')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (b) u_y vs u_z (lateral displacement growth)
ax = axes[1]
ax.plot(-uz_history, np.abs(uy_history), 'g-', lw=1.5)
ax.axvline(-uz_at_Pmax, color='r', ls='--', lw=0.8, alpha=0.5, label='Limit point')
ax.set_xlabel('−u_z [mm]')
ax.set_ylabel('|u_y| [mm] (lateral sway)')
ax.set_title('Lateral displacement growth')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (c) P vs u_y (load vs lateral displacement — classic LTB curve)
ax = axes[2]
ax.plot(np.abs(uy_history), load_history/1000, 'r-', lw=1.5)
ax.plot(abs(uy_at_Pmax), P_max/1000, 'ro', markersize=8, zorder=5,
        label=f'P_max = {P_max/1000:.2f} kN')
ax.set_xlabel('|u_y| [mm] (lateral sway)')
ax.set_ylabel('P [kN]')
ax.set_title('P vs lateral displacement (LTB)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Analytical comparison ---
C1  = 1.365
Iw  = Iz_cs * d**2 / 4
M_cr = C1 * (np.pi / L) * np.sqrt(E * Iz_cs * G * J_cs) * \
       np.sqrt(1 + (np.pi**2 * E * Iw) / (G * J_cs * L**2))
P_cr_analytical = 4 * M_cr / L

print(f"\nComparison:")
print(f"  P_max  (NL imperfect) = {P_max/1000:.2f} kN")
print(f"  P_cr   (analytical LTB) = {P_cr_analytical/1000:.2f} kN")
print(f"  Ratio  P_max / P_cr    = {P_max/P_cr_analytical:.3f}")
print(f"\n  Imperfection L/{int(L/imp_amplitude)} reduces the capacity by "
      f"{(1 - P_max/P_cr_analytical)*100:.1f}%")

In [ ]:
# --- 3D deformed shape at the limit point and final state ---

conn_idx = np.array([[tag_to_idx[int(n)] for n in row] for row in connectivity])

fig = plt.figure(figsize=(16, 10))

# Deformation at the final state (last converged step)
phi = disp_final[:, :3]
max_d = np.max(np.abs(phi))
scale = 0.08 * L / max_d if max_d > 1e-15 else 1.0

deformed = node_coords_imp + scale * phi   # from imperfect base

# Color by lateral displacement (uy)
uy_all = phi[:, 1]
uy_range = max(np.abs(uy_all).max(), 1e-12)
uy_norm = (uy_all + uy_range) / (2 * uy_range)  # 0 to 1

# (a) 3D perspective
ax1 = fig.add_subplot(221, projection='3d')
for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    avg_c = np.mean(uy_norm[row])
    ax1.plot(quad[:, 0], quad[:, 1], quad[:, 2],
             color=plt.cm.RdBu_r(avg_c), lw=0.3)

ax1.set_title(f'Deformed shape at P={load_history[-1]/1000:.1f} kN (×{scale:.0f})', fontsize=10)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.view_init(elev=25, azim=-60)

# (b) Top view (lateral buckling visible)
ax2 = fig.add_subplot(222, projection='3d')
for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    avg_c = np.mean(uy_norm[row])
    ax2.plot(quad[:, 0], quad[:, 1], quad[:, 2],
             color=plt.cm.RdBu_r(avg_c), lw=0.3)

ax2.set_title('Top view — lateral sway', fontsize=10)
ax2.view_init(elev=85, azim=-90)
ax2.set_xlabel('X'); ax2.set_ylabel('Y')

# (c) Front view at midspan
ax3 = fig.add_subplot(223, projection='3d')
for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    avg_c = np.mean(uy_norm[row])
    ax3.plot(quad[:, 0], quad[:, 1], quad[:, 2],
             color=plt.cm.RdBu_r(avg_c), lw=0.3)

ax3.set_title('End view — cross-section twist', fontsize=10)
ax3.view_init(elev=0, azim=0)
ax3.set_ylabel('Y'); ax3.set_zlabel('Z')

# (d) Perfect vs imperfect vs buckled — midspan cross-section
ax4 = fig.add_subplot(224)
tol_x = 60
mid_mask = np.abs(node_coords[:, 0] - L/2) < tol_x

ax4.plot(node_coords[mid_mask, 1], node_coords[mid_mask, 2],
         'b.', ms=2, alpha=0.3, label='Perfect')
ax4.plot(node_coords_imp[mid_mask, 1], node_coords_imp[mid_mask, 2],
         'g.', ms=2, alpha=0.4, label=f'Imperfect (L/{int(L/imp_amplitude)})')

buckled_coords = node_coords_imp + phi
ax4.plot(buckled_coords[mid_mask, 1], buckled_coords[mid_mask, 2],
         'r.', ms=3, alpha=0.6, label='Buckled (true scale)')

ax4.set_xlabel('Y [mm]')
ax4.set_ylabel('Z [mm]')
ax4.set_title('Midspan cross-section evolution')
ax4.legend(fontsize=8)
ax4.set_aspect('equal')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5 — Imperfection Sensitivity (Optional Extension)

To study how the collapse load depends on the imperfection amplitude,
re-run the analysis with different values of `imp_amplitude`:

| Imperfection | Typical P_max / P_cr |
|-------------|---------------------|
| L/2000      | ~0.95               |
| L/1000      | ~0.90               |
| L/500       | ~0.82               |
| L/200       | ~0.70               |

This table varies with cross-section, slenderness, and load type.
The shell model computes each case directly — no need for reduction
factor charts.

To add **material nonlinearity** (plasticity), replace
`ElasticMembranePlateSection` with a fiber-based `PlateFiber` section
using `J2Plasticity` or `Steel02` materials through the thickness.

## Part 6 — Gmsh Post-processing Views

In [ ]:
# --- Gmsh views ---
node_tag_list = list(node_tags.astype(int))

# Final deformed shape (translational DOFs)
g.view.add_node_vector("Deformed shape (final)", node_tag_list, disp_final[:, :3])

# Lateral displacement (scalar — smooth contour)
g.view.add_node_scalar("|u_y| lateral sway", node_tag_list, np.abs(disp_final[:, 1]))

# Vertical displacement
g.view.add_node_scalar("u_z vertical", node_tag_list, disp_final[:, 2])

# Imperfection shape
g.view.add_node_vector("Imperfection (LTB mode)", node_tag_list,
                       scale_factor * phi_trans)

# Displacement magnitude
mag = np.sqrt(np.sum(disp_final[:, :3]**2, axis=1))
g.view.add_node_scalar("|u| total displacement", node_tag_list, mag)

print(f"Created {g.view.count()} Gmsh views")

## Part 7 — Launch Gmsh GUI & Finalize

In [ ]:
# Explore the buckled shape interactively.
# Tips:
#   View > select "Deformed shape" > Displacement Factor slider
#   Tools > Options > View > Vector Display → Displacement
g.launch_gui()

In [ ]:
g.finalize()